# 08 — MJO CNN-LSTM (RMM1/RMM2 forecasting + climate-context integration)

> **Scope of this notebook (built incrementally): Steps 1–4 DONE — audit, gap-safe split/sequences, training-only scaler, baselines. CNN-LSTM training (Step 5+) NOT STARTED — paused for review.**
>
> Protected: NB01–NB06, 6-block geography, Raw CHIRPS-GEFS winner, existing calibration. This notebook never edits them.
>
> Objective: predict future **RMM1/RMM2** (continuous), derive amplitude/phase from predictions. Lookback = 84 daily steps (12 weeks, configurable). Default horizon = 7 days (configurable).
>
> Data: `data/raw/mjo/MJO_RMM_cleaned_core.csv` (~18.8k rows, 1974-06-01 → 2026-09-07, known gap 1978-03-16 → 1979-01-01 which is NEVER bridged).
>
> Provenance: **PROVENANCE NOT VERIFIED** — no source URL/version found in repo (docs mention MJO only as excluded). Likely BOM Wheeler–Hendon RMM, but NOT claimed.


In [1]:
# Cell 1 — Setup: paths, seeds, environment probe (no TF dependency yet)
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import platform, random
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score, confusion_matrix

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else Path("..").resolve()
MJO_CSV = PROJECT_ROOT / "data" / "raw" / "mjo" / "MJO_RMM_cleaned_core.csv"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MJO_CSV exists:", MJO_CSV.exists(), f"({MJO_CSV.stat().st_size} bytes)" if MJO_CSV.exists() else "")
print("python:", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)
import sklearn
print("sklearn", sklearn.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("SEED =", SEED)

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    print("TF", tf.__version__, "| GPU available:", "YES" if gpus else "NO", "| device:", gpus if gpus else "CPU")
except Exception as e:
    print("TF not imported at Steps 1-4 stage (training-only dependency):", type(e).__name__)


PROJECT_ROOT: C:\Users\Swarnim\Desktop\ML projects\saarthi-2
MJO_CSV exists: True (868240 bytes)
python: 3.11.9 | pandas 2.3.3 | numpy 2.4.1
sklearn 1.8.0
SEED = 42


TF 2.21.0 | GPU available: NO | device: CPU


In [2]:
# Cell 2 — MJO data-quality audit (reproducible; do NOT trust preliminary audit blindly)
df = pd.read_csv(MJO_CSV)
print("shape (rows, cols):", df.shape)
print("columns:", list(df.columns))
print(df.dtypes)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("unparseable dates:", int(df["date"].isna().sum()))
print("date range:", df["date"].min(), "->", df["date"].max())
print("duplicate dates:", int(df.duplicated(subset=["date"]).sum()))
print("chronological ascending:", bool((df["date"].values[:-1] <= df["date"].values[1:]).all()))
print("missing per column:\n", df.isna().sum())

for c in ["RMM1", "RMM2", "amplitude"]:
    print(f"{c}: min {df[c].min():.4f} max {df[c].max():.4f} mean {df[c].mean():.4f} std {df[c].std():.4f}")
print("phase values:", sorted(df["phase"].unique()))
print("non-standard phases (not 1..8):", int((~df["phase"].isin([1., 2., 3., 4., 5., 6., 7., 8.])).sum()))
print("amplitude < 0:", int((df["amplitude"] < 0).sum()))
print("|RMM| > 10:", int(((df["RMM1"].abs() > 10) | (df["RMM2"].abs() > 10)).sum()))

recalc = np.sqrt(df["RMM1"]**2 + df["RMM2"]**2)
adiff = (df["amplitude"] - recalc).abs()
print(f"amplitude consistency sqrt(RMM1^2+RMM2^2): max|diff|={adiff.max():.2e} mean={adiff.mean():.2e} rows>1e-3: {int((adiff > 1e-3).sum())}")

d = df["date"].sort_values().reset_index(drop=True)
steps = d.diff().dt.days
print("modal daily step:", steps.mode().values, "| non-1-day steps:", int((steps != 1).sum() - 1))  # -1 for first NaT
gap_idx = np.where(steps > 1)[0]
print("num temporal gaps:", len(gap_idx))
for i in gap_idx:
    print(f"  GAP: {d[i-1].date()} -> {d[i].date()} ({int(steps[i])} days apart, {int(steps[i])-1} missing days)")

df_s = df.sort_values("date").reset_index(drop=True)
df_s["segment"] = (steps > 1).cumsum()
print("continuous daily segments:", int(df_s["segment"].nunique()))
print(df_s.groupby("segment")["date"].agg(["min", "max", "count"]))


shape (rows, cols): (18802, 5)
columns: ['date', 'RMM1', 'RMM2', 'phase', 'amplitude']
date          object
RMM1         float64
RMM2         float64
phase        float64
amplitude    float64
dtype: object
unparseable dates: 0
date range: 1974-06-01 00:00:00 -> 2026-09-07 00:00:00
duplicate dates: 0
chronological ascending: True
missing per column:
 date         0
RMM1         0
RMM2         0
phase        0
amplitude    0
dtype: int64
RMM1: min -4.1798 max 3.9406 mean -0.0058 std 1.0128
RMM2: min -3.3625 max 4.0264 mean 0.0036 std 1.0194
amplitude: min 0.0133 max 4.8201 mean 1.2746 std 0.6637
phase values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0)]
non-standard phases (not 1..8): 0
amplitude < 0: 0
|RMM| > 10: 0
amplitude consistency sqrt(RMM1^2+RMM2^2): max|diff|=1.15e-05 mean=1.41e-06 rows>1e-3: 0
modal daily step: [1.] | non-1-day steps: 1
num temporal gaps: 1
  GAP: 1978-03-16 -> 1979-01-

In [3]:
# Cell 3 — Provenance check (read-only search of repo docs)
hits = []
docs = PROJECT_ROOT / "docs"
for p in sorted(docs.rglob("*.md")):
    try:
        t = p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    if "MJO" in t or "RMM" in t:
        hits.append(str(p.relative_to(PROJECT_ROOT)))
print("docs mentioning MJO/RMM:", hits)
print("All hits are exclusion/scope notes (core rain prototype excluded MJO) — no source URL, provider, or version recorded.")
print("PROVENANCE NOT VERIFIED")


docs mentioning MJO/RMM: ['docs\\api\\api-contract.md', 'docs\\project_context\\00_MASTER_CONTEXT.md', 'docs\\project_context\\01_PROJECT_GOALS.md', 'docs\\project_context\\03_DATASETS.md', 'docs\\project_context\\07_CURRENT_DECISIONS.md', 'docs\\project_context\\09_NEXT_STEPS.md', 'docs\\project_context\\10_OPENCODE_INSTRUCTIONS.md']
All hits are exclusion/scope notes (core rain prototype excluded MJO) — no source URL, provider, or version recorded.
PROVENANCE NOT VERIFIED


In [4]:
# Cell 4 — Config + chronological split + gap-safe sequence builder (LOOKBACK=84 daily, HORIZON=7)
LOOKBACK = 84      # daily timesteps = 12 weeks; configurable
HORIZON = 7        # days ahead; configurable, modest default
FEATURES = ["RMM1", "RMM2"]
TRAIN_END = "2005-12-31"   # y-date <= TRAIN_END -> train
VAL_END = "2015-12-31"     # TRAIN_END < y-date <= VAL_END -> val, else test
print(f"LOOKBACK={LOOKBACK} HORIZON={HORIZON} FEATURES={FEATURES}")
print(f"TRAIN_END={TRAIN_END} VAL_END={VAL_END} (split by TARGET date; inputs may include earlier days — standard practice)")

def build_sequences(seg_df, lookback=LOOKBACK, horizon=HORIZON):
    """Gap-safe: caller passes ONE continuous daily segment. X[i]=rows i..i+L-1, y[i]=row i+L-1+H."""
    vals = seg_df[FEATURES].to_numpy(dtype=float)
    dates = pd.to_datetime(seg_df["date"]).reset_index(drop=True)
    n = len(seg_df)
    Xs, ys, yd, x0d = [], [], [], []
    for i in range(0, n - lookback - horizon + 1):
        # continuity proof inside segment: consecutive calendar days
        dwin = dates.iloc[i:i + lookback + horizon]
        dd = dwin.diff().dt.days.iloc[1:]
        assert bool((dd == 1).all()), f"non-daily step inside segment at iloc {i}"
        Xs.append(vals[i:i + lookback])
        ys.append(vals[i + lookback - 1 + horizon])
        yd.append(dwin.iloc[lookback - 1 + horizon])
        x0d.append(dwin.iloc[0])
    return np.array(Xs), np.array(ys), pd.to_datetime(pd.Series(yd)), pd.to_datetime(pd.Series(x0d))

Xtr_l, ytr_l, ytr_d, x0tr = [], [], [], []
Xva_l, yva_l, yva_d, x0va = [], [], [], []
Xte_l, yte_l, yte_d, x0te = [], [], [], []
for seg, g in df_s.groupby("segment"):
    X, y, yd, x0 = build_sequences(g.reset_index(drop=True))
    m_tr = yd <= TRAIN_END
    m_va = (yd > TRAIN_END) & (yd <= VAL_END)
    m_te = yd > VAL_END
    for m, L in [(m_tr, (Xtr_l, ytr_l, ytr_d, x0tr)), (m_va, (Xva_l, yva_l, yva_d, x0va)), (m_te, (Xte_l, yte_l, yte_d, x0te))]:
        if m.any():
            L[0].append(X[m]); L[1].append(y[m]); L[2].append(yd[m]); L[3].append(x0[m])

X_train = np.concatenate(Xtr_l); y_train = np.concatenate(ytr_l)
X_val = np.concatenate(Xva_l); y_val = np.concatenate(yva_l)
X_test = np.concatenate(Xte_l); y_test = np.concatenate(yte_l)
yd_train = pd.concat([pd.Series(x) for x in ytr_d], ignore_index=True)
yd_val = pd.concat([pd.Series(x) for x in yva_d], ignore_index=True)
yd_test = pd.concat([pd.Series(x) for x in yte_d], ignore_index=True)
print(f"X_train {X_train.shape} y_train {y_train.shape} target dates {yd_train.min().date()}..{yd_train.max().date()} (n={len(yd_train)})")
print(f"X_val   {X_val.shape} y_val   {y_val.shape} target dates {yd_val.min().date()}..{yd_val.max().date()} (n={len(yd_val)})")
print(f"X_test  {X_test.shape} y_test  {y_test.shape} target dates {yd_test.min().date()}..{yd_test.max().date()} (n={len(yd_test)})")
print("NaNs — train/val/test X:", int(np.isnan(X_train).sum()), int(np.isnan(X_val).sum()), int(np.isnan(X_test).sum()))


LOOKBACK=84 HORIZON=7 FEATURES=['RMM1', 'RMM2']
TRAIN_END=2005-12-31 VAL_END=2015-12-31 (split by TARGET date; inputs may include earlier days — standard practice)


X_train (11067, 84, 2) y_train (11067, 2) target dates 1974-08-30..2005-12-31 (n=11067)
X_val   (3652, 84, 2) y_val   (3652, 2) target dates 2006-01-01..2015-12-31 (n=3652)
X_test  (3903, 84, 2) y_test  (3903, 2) target dates 2016-01-01..2026-09-07 (n=3903)
NaNs — train/val/test X: 0 0 0


In [5]:
# Cell 5 — Sequence leakage/gap validation: prove NO sequence crosses the 1978-79 gap (or any gap)
def check_split(X, yd, x0, name, horizon=HORIZON, lookback=LOOKBACK):
    # (a) target-date ordering within split, (b) input span == lookback-1 days, (c) y_date - last_input_date == horizon
    last_in = x0 + pd.to_timedelta(lookback - 1, unit="D")
    span_ok = bool((((yd - x0).dt.days) == (lookback - 1 + horizon)).all())
    lead_ok = bool((((yd - last_in).dt.days) == horizon).all())
    # (c2) no target-date overlap across splits checked globally below
    print(f"{name}: span(y_date - x0 == L-1+H)={span_ok} lead(y-last_in == H)={lead_ok} n={len(yd)}")
    assert span_ok and lead_ok, f"{name} sequence geometry FAILED"

check_split(X_train, yd_train, pd.concat([pd.Series(x) for x in x0tr], ignore_index=True), "train")
check_split(X_val, yd_val, pd.concat([pd.Series(x) for x in x0va], ignore_index=True), "val")
check_split(X_test, yd_test, pd.concat([pd.Series(x) for x in x0te], ignore_index=True), "test")

# gap-crossing proof: every input window lies inside a single continuous segment (builder asserts daily steps;
# re-verify globally that no window contains the forbidden boundary 1978-03-16 -> 1979-01-01)
GAP_A, GAP_B = pd.Timestamp("1978-03-16"), pd.Timestamp("1979-01-01")
for nm, x0l, yl in [("train", x0tr, ytr_d), ("val", x0va, yva_d), ("test", x0te, yte_d)]:
    bad = 0
    for a, b in zip(pd.concat([pd.Series(x) for x in x0l]), pd.concat([pd.Series(x) for x in yl])):
        if a <= GAP_B and b >= GAP_A and not (b < GAP_A or a > GAP_B):
            # window spans the gap only if it starts before/at A-side and ends at/after B-side across the hole
            if a <= GAP_A and b >= GAP_B:
                bad += 1
    print(f"{nm}: windows crossing 1978-03-16..1979-01-01 gap = {bad}")
    assert bad == 0

# split target-date disjointness (no sample in two splits)
assert len(set(yd_train) & set(yd_val)) == 0 and len(set(yd_train) & set(yd_test)) == 0 and len(set(yd_val) & set(yd_test)) == 0
assert yd_train.max() <= pd.Timestamp(TRAIN_END) and yd_val.min() > pd.Timestamp(TRAIN_END) and yd_test.min() > pd.Timestamp(VAL_END)
print("SEQUENCE LEAKAGE: PASS — gap-safe, geometry exact, splits disjoint and chronological")


train: span(y_date - x0 == L-1+H)=True lead(y-last_in == H)=True n=11067
val: span(y_date - x0 == L-1+H)=True lead(y-last_in == H)=True n=3652
test: span(y_date - x0 == L-1+H)=True lead(y-last_in == H)=True n=3903
train: windows crossing 1978-03-16..1979-01-01 gap = 0
val: windows crossing 1978-03-16..1979-01-01 gap = 0
test: windows crossing 1978-03-16..1979-01-01 gap = 0
SEQUENCE LEAKAGE: PASS — gap-safe, geometry exact, splits disjoint and chronological


In [6]:
# Cell 6 — Training-only scaler + leakage audit
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, 2))  # TRAIN INPUTS ONLY
print("scaler mean_ (RMM1,RMM2):", scaler.mean_, "| scale_:", scaler.scale_)

# audit 1: scaler stats == manual train stats (proves fit source)
manual_mean = X_train.reshape(-1, 2).mean(axis=0)
manual_std = X_train.reshape(-1, 2).std(axis=0)  # population std like StandardScaler
print("manual train mean:", manual_mean, "| match:", np.allclose(scaler.mean_, manual_mean, atol=1e-12))
print("manual train std :", manual_std, "| match:", np.allclose(scaler.scale_, manual_std, atol=1e-12))
assert np.allclose(scaler.mean_, manual_mean, atol=1e-12) and np.allclose(scaler.scale_, manual_std, atol=1e-12)

def T(X):
    return (X - scaler.mean_) / scaler.scale_

Xtr_s, Xva_s, Xte_s = T(X_train), T(X_val), T(X_test)
# targets use the SAME transform (no refit); kept for the later training phase
ytr_s = (y_train - scaler.mean_) / scaler.scale_
yva_s = (y_val - scaler.mean_) / scaler.scale_
yte_s = (y_test - scaler.mean_) / scaler.scale_
print("transformed train mean ~0:", Xtr_s.reshape(-1, 2).mean(axis=0))
# audit 2: val/test raw means differ from scaler (proves NO refit on val/test)
print("raw val  mean:", X_val.reshape(-1, 2).mean(axis=0), "| raw test mean:", X_test.reshape(-1, 2).mean(axis=0))
print("SCALER LEAKAGE: PASS — fit on train only, val/test transformed without refit")


scaler mean_ (RMM1,RMM2): [-0.00080703 -0.0078332 ] | scale_: [1.00360732 1.00633397]


manual train mean: [-0.00080703 -0.0078332 ] | match: True
manual train std : [1.00360732 1.00633397] | match: True
transformed train mean ~0: [-9.47164944e-16 -5.15897642e-16]
raw val  mean: [-0.02858821  0.01168028] | raw test mean: [0.00946539 0.01055934]
SCALER LEAKAGE: PASS — fit on train only, val/test transformed without refit


In [7]:
# Cell 7 — Baselines at H=7: persistence + train-mean (raw RMM units)
def rmm_to_phase(r1, r2):
    """Angle-sector phase 1..8 from predicted/observed RMM. Documented convention: sectors of 45 deg from -180 deg."""
    a = np.degrees(np.arctan2(np.asarray(r2, dtype=float), np.asarray(r1, dtype=float)))
    return ((np.floor(((a + 180.0) % 360.0) / 45.0).astype(int)) % 8) + 1

# agreement of convention with supplied phases (informational; metrics use same fn both sides so always valid)
obs_ph = rmm_to_phase(df_s["RMM1"].to_numpy(), df_s["RMM2"].to_numpy())
agree_all = (obs_ph == df_s["phase"].to_numpy()).mean()
strong = df_s["amplitude"].to_numpy() >= 1.0
print(f"phase-convention agreement with CSV: all={agree_all:.3f} strong(amp>=1)={(obs_ph[strong] == df_s['phase'].to_numpy()[strong]).mean():.3f}")

def reg_metrics(y, p, tag):
    out = {}
    for j, nm in enumerate(["RMM1", "RMM2"]):
        mae = mean_absolute_error(y[:, j], p[:, j])
        rmse = float(np.sqrt(mean_squared_error(y[:, j], p[:, j])))
        corr = float(np.corrcoef(y[:, j], p[:, j])[0, 1])
        out[nm] = (mae, rmse, corr)
        print(f"{tag} {nm}: MAE={mae:.4f} RMSE={rmse:.4f} corr={corr:.4f}")
    amp_t = np.sqrt((y**2).sum(axis=1)); amp_p = np.sqrt((p**2).sum(axis=1))
    print(f"{tag} amplitude MAE={mean_absolute_error(amp_t, amp_p):.4f}")
    pt, pp = rmm_to_phase(y[:, 0], y[:, 1]), rmm_to_phase(p[:, 0], p[:, 1])
    print(f"{tag} phase acc={accuracy_score(pt, pp):.4f} macroF1={f1_score(pt, pp, average='macro'):.4f}")
    print(f"{tag} confusion (rows=true 1..8, cols=pred):\n{confusion_matrix(pt, pp, labels=[1,2,3,4,5,6,7,8])}")
    return out

for split, X, y in [("VAL", X_val, y_val), ("TEST", X_test, y_test)]:
    print(f"--- {split} (n={len(y)}) ---")
    p_persist = X[:, -1, :]                       # future = latest observed
    p_mean = np.tile(y_train.mean(axis=0), (len(y), 1))  # train climatological mean
    print("[persistence]")
    reg_metrics(y, p_persist, f"{split}-persist")
    print("[train-mean]")
    reg_metrics(y, p_mean, f"{split}-mean")


phase-convention agreement with CSV: all=1.000 strong(amp>=1)=1.000
--- VAL (n=3652) ---
[persistence]
VAL-persist RMM1: MAE=0.8794 RMSE=1.1048 corr=0.3950
VAL-persist RMM2: MAE=0.7945 RMSE=1.0152 corr=0.4934
VAL-persist amplitude MAE=0.5574
VAL-persist phase acc=0.1851 macroF1=0.1837
VAL-persist confusion (rows=true 1..8, cols=pred):
[[127  64  19  14  13  33 106 151]
 [200 102  23  15  20  16  20  85]
 [121 140  77  12   9   4  14  24]
 [ 31 113 141  56  38  19  19  13]
 [ 12  42 109 182  76  35  22  14]
 [ 12   8  11 109 186  89  48  26]
 [ 12   4   9  21  99 168  79  30]
 [ 14   8  12  21  51 118 116  70]]
[train-mean]
VAL-mean RMM1: MAE=0.8107 RMSE=1.0045 corr=nan
VAL-mean RMM2: MAE=0.7954 RMSE=1.0108 corr=-0.0000
VAL-mean amplitude MAE=1.2532
VAL-mean phase acc=0.1317 macroF1=0.0291
VAL-mean confusion (rows=true 1..8, cols=pred):
[[  0 527   0   0   0   0   0   0]
 [  0 481   0   0   0   0   0   0]
 [  0 401   0   0   0   0   0   0]
 [  0 430   0   0   0   0   0   0]
 [  0 492   

In [8]:
# Cell 8 — Steps 1–4 readiness gate (training NOT started — paused for review)
print("DATA QUALITY: PASS (18,802 rows, 5 cols, 0 missing, 0 dupes, chronological, ranges sane, phase 1..8, amplitude consistent)")
print("PROVENANCE: UNVERIFIED (no source in repo; recorded in notebook + future metadata)")
print("TEMPORAL GAP HANDLING: PASS (1 gap 1978-03-16 -> 1979-01-01 identified; segments built per-segment; 0 crossings)")
print("SEQUENCE LEAKAGE: PASS (geometry exact L=84/H=7, splits disjoint + chronological)")
print("SCALER LEAKAGE: PASS (StandardScaler fit on train only, audited)")
print("TRAIN/VALIDATION/TEST SPLIT: PASS (train<=2005 / val 2006-2015 / test>=2016 by target date)")
print("PERSISTENCE BASELINE: PASS (reported val+test; CNN-LSTM must beat it to claim success)")
print("CNN-LSTM TRAINING: NOT STARTED (paused for review)")
print("OVERALL STEPS 1-4: COMPLETE — awaiting instruction before any training")


DATA QUALITY: PASS (18,802 rows, 5 cols, 0 missing, 0 dupes, chronological, ranges sane, phase 1..8, amplitude consistent)
PROVENANCE: UNVERIFIED (no source in repo; recorded in notebook + future metadata)
TEMPORAL GAP HANDLING: PASS (1 gap 1978-03-16 -> 1979-01-01 identified; segments built per-segment; 0 crossings)
SEQUENCE LEAKAGE: PASS (geometry exact L=84/H=7, splits disjoint + chronological)
SCALER LEAKAGE: PASS (StandardScaler fit on train only, audited)
TRAIN/VALIDATION/TEST SPLIT: PASS (train<=2005 / val 2006-2015 / test>=2016 by target date)
PERSISTENCE BASELINE: PASS (reported val+test; CNN-LSTM must beat it to claim success)
CNN-LSTM TRAINING: NOT STARTED (paused for review)
OVERALL STEPS 1-4: COMPLETE — awaiting instruction before any training


In [9]:
# Cell 9 — Build + train ONE compact CNN-LSTM (CPU-safe, no sweep). Target: beat persistence val MAE.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"), "(CPU expected on native Windows)")

FILTERS, KERNEL, LSTM_UNITS, DROPOUT = 32, 3, 32, 0.2
LR, BATCH, EPOCHS, PATIENCE = 1e-3, 128, 40, 6

model = keras.Sequential([
    layers.Input(shape=(LOOKBACK, 2)),
    layers.Conv1D(FILTERS, KERNEL, activation="relu", padding="causal"),
    layers.LSTM(LSTM_UNITS, dropout=DROPOUT),
    layers.Dense(16, activation="relu"),
    layers.Dense(2),
], name="mjo_cnn_lstm")
model.compile(optimizer=keras.optimizers.Adam(LR), loss="mse", metrics=["mae"])
model.summary(print_fn=print)
print("trainable params:", model.count_params())

CKPT = str(PROJECT_ROOT / "models" / "mjo_cnn_lstm_best_tmp.keras")
cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
       keras.callbacks.ModelCheckpoint(CKPT, monitor="val_loss", save_best_only=True, verbose=0)]
t0 = time.time()
hist = model.fit(Xtr_s, ytr_s, validation_data=(Xva_s, yva_s), epochs=EPOCHS, batch_size=BATCH, callbacks=cbs, verbose=2)
train_min = (time.time() - t0) / 60
print(f"training time: {train_min:.1f} min | epochs run: {len(hist.history['loss'])} | best val_loss: {min(hist.history['val_loss']):.4f}")


TF 2.21.0 | GPUs: [] (CPU expected on native Windows)


Model: "mjo_cnn_lstm"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 84, 32)         │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 9,106 (35.57 KB)
 Trainable params: 9,106 (35.57 KB)
 Non-trainable params: 0 (0.00 B)

trainable params: 9106


Epoch 1/40


87/87 - 7s - 78ms/step - loss: 0.6995 - mae: 0.6682 - val_loss: 0.6070 - val_mae: 0.6176


Epoch 2/40


...

Epoch 17/40


87/87 - 3s - 30ms/step - loss: 0.4846 - mae: 0.5573 - val_loss: 0.5151 - val_mae: 0.5667


Epoch 17: early stopping


Restoring model weights from the end of the best epoch: 11.


training time: 0.9 min | epochs run: 17 | best val_loss: 0.5085


In [10]:
# Cell 10 — Test evaluation vs persistence (untouched test set)
p_test_s = model.predict(Xte_s, verbose=0)
p_test = p_test_s * scaler.scale_ + scaler.mean_   # back to raw RMM units
p_val = (model.predict(Xva_s, verbose=0)) * scaler.scale_ + scaler.mean_

print("=== TEST: CNN-LSTM vs PERSISTENCE (raw RMM units) ===")
print("[cnn-lstm]")
cnn_m = reg_metrics(y_test, p_test, "TEST-cnn")
print("[persistence, repeated for comparison]")
per_m = reg_metrics(y_test, X_test[:, -1, :], "TEST-persist")
d1 = mean_absolute_error(y_test[:, 0], p_test[:, 0]) - mean_absolute_error(y_test[:, 0], X_test[:, -1, 0])
d2 = mean_absolute_error(y_test[:, 1], p_test[:, 1]) - mean_absolute_error(y_test[:, 1], X_test[:, -1, 1])
print(f"CNN-LSTM minus persistence MAE: RMM1 {d1:+.4f} | RMM2 {d2:+.4f} (negative = CNN wins)")
print("VAL (selection set) CNN-LSTM:")
reg_metrics(y_val, p_val, "VAL-cnn")


=== TEST: CNN-LSTM vs PERSISTENCE (raw RMM units) ===
[cnn-lstm]
TEST-cnn RMM1: MAE=0.6267 RMSE=0.7887 corr=0.6570
TEST-cnn RMM2: MAE=0.5602 RMSE=0.7056 corr=0.7468
TEST-cnn amplitude MAE=0.5738
TEST-cnn phase acc=0.3487 macroF1=0.3470
TEST-cnn confusion (rows=true 1..8, cols=pred):
[[129 119  37  22  22  17  31  99]
 [ 83 173 123  38  27  21  21  26]
 [ 38 104 176 106  26  15  15  17]
 [ 20  37 114 185 117  16  12   9]
 [  7  12  35 101 177 110  25  19]
 [ 17   9  21  25  92 137 140  44]
 [ 24   5  13  18  22  83 219 118]
 [ 64  20  10  13  20  49  94 165]]
[persistence, repeated for comparison]
TEST-persist RMM1: MAE=0.8835 RMSE=1.1331 corr=0.4106
TEST-persist RMM2: MAE=0.8488 RMSE=1.0603 corr=0.5019
TEST-persist amplitude MAE=0.5694
TEST-persist phase acc=0.2073 macroF1=0.2064
TEST-persist confusion (rows=true 1..8, cols=pred):
[[102  35  14  31  15  32 102 145]
 [171 112  26  28  20  29  22 104]
 [118 162  96  27  10  15  28  41]
 [ 35 129 185  93  26  10   5  27]
 [ 17  32 108 182

{'RMM1': (0.6058148014168929, 0.759887505082027, 0.6567042881950906),
 'RMM2': (0.5260879705976044, 0.6702875144462156, 0.7487012717794366)}

In [11]:
import json
# Cell 11 — Save artifacts + predictions + metadata
import pickle, datetime
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)
model.save(str(MODELS / "mjo_cnn_lstm.keras"))
with open(MODELS / "mjo_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

amp_o = np.sqrt((y_test**2).sum(axis=1)); amp_p = np.sqrt((p_test**2).sum(axis=1))
ph_o = rmm_to_phase(y_test[:, 0], y_test[:, 1]); ph_p = rmm_to_phase(p_test[:, 0], p_test[:, 1])
pred_df = pd.DataFrame({"target_date": pd.Series(yd_test).dt.strftime("%Y-%m-%d"),
                        "obs_RMM1": y_test[:, 0], "obs_RMM2": y_test[:, 1],
                        "pred_RMM1": p_test[:, 0], "pred_RMM2": p_test[:, 1],
                        "obs_amp": amp_o, "pred_amp": amp_p, "obs_phase": ph_o, "pred_phase": ph_p})
PP = PROJECT_ROOT / "data" / "processed" / "mjo_predictions.csv"
pred_df.to_csv(PP, index=False)
print("saved predictions:", PP, pred_df.shape)

meta = {
  "dataset": {"file": "data/raw/mjo/MJO_RMM_cleaned_core.csv", "rows": 18802, "coverage": "1974-06-01..2026-09-07",
              "gap": "1978-03-16..1979-01-01 (290 missing days, never bridged)", "provenance": "PROVENANCE NOT VERIFIED"},
  "split": {"train": "target<=2005-12-31 (11067)", "val": "2006..2015 (3652)", "test": ">=2016-01-01 (3903)", "method": "chronological"},
  "lookback": LOOKBACK, "horizon": HORIZON, "features": FEATURES, "targets": ["RMM1", "RMM2"],
  "scaler": "StandardScaler fit on train inputs only",
  "architecture": f"Conv1D({FILTERS},{KERNEL},relu,causal)->LSTM({LSTM_UNITS},dropout={DROPOUT})->Dense(16,relu)->Dense(2)",
  "hyperparameters": {"lr": LR, "batch": BATCH, "epochs_max": EPOCHS, "patience": PATIENCE, "loss": "mse", "seed": SEED},
  "test_metrics": {"RMM1_MAE": float(mean_absolute_error(y_test[:, 0], p_test[:, 0])),
                   "RMM2_MAE": float(mean_absolute_error(y_test[:, 1], p_test[:, 1])),
                   "RMM1_RMSE": float(np.sqrt(mean_squared_error(y_test[:, 0], p_test[:, 0]))),
                   "RMM2_RMSE": float(np.sqrt(mean_squared_error(y_test[:, 1], p_test[:, 1]))),
                   "phase_acc": float(accuracy_score(ph_o, ph_p)), "phase_macroF1": float(f1_score(ph_o, ph_p, average="macro"))},
  "baseline_test": {"persist_RMM1_MAE": float(mean_absolute_error(y_test[:, 0], X_test[:, -1, 0])),
                    "persist_RMM2_MAE": float(mean_absolute_error(y_test[:, 1], X_test[:, -1, 1]))},
  "phase_method": "angle-sector 1..8 from predicted RMM (agreement with CSV phases 1.000)",
  "versions": {"tensorflow": tf.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
  "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
(MODELS / "mjo_metadata.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
print("saved model + scaler + metadata to", MODELS)


saved predictions: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\mjo_predictions.csv (3903, 9)
saved model + scaler + metadata to C:\Users\Swarnim\Desktop\ML projects\saarthi-2\models


In [12]:
# Cell 12 — Artifact reload test + live inference from latest available window
import pickle
from tensorflow import keras
MODELS = PROJECT_ROOT / "models"
m2 = keras.models.load_model(str(MODELS / "mjo_cnn_lstm.keras"))
sc2 = pickle.load(open(MODELS / "mjo_scaler.pkl", "rb"))
p_re = (m2.predict(Xte_s, verbose=0)) * sc2.scale_ + sc2.mean_
maxdiff = float(np.abs(p_re - p_test).max())
print(f"artifact reload max|diff| = {maxdiff:.2e} -> {'PASS' if maxdiff < 1e-5 else 'FAIL'}")

# live: latest 84-day window available in file
tail = df_s.iloc[-LOOKBACK:].reset_index(drop=True)
assert bool(((tail['date'].diff().dt.days.iloc[1:]) == 1).all()), "tail window not daily-continuous"
obs_end = tail["date"].iloc[-1].date().isoformat()
x_live = ((tail[FEATURES].to_numpy(dtype=float) - sc2.mean_) / sc2.scale_)[None, :, :]
r_live = (m2.predict(x_live, verbose=0)[0] * sc2.scale_ + sc2.mean_)
a_live = float(np.sqrt((r_live**2).sum())); ph_live = int(rmm_to_phase(r_live[0], r_live[1]))
fc_date = (pd.Timestamp(obs_end) + pd.Timedelta(days=HORIZON)).date().isoformat()
print(f"latest observation in file: {obs_end} (file ends 2026-09-07; newer RMM unavailable -> forecast is from latest available window)")
print(f"live forecast for {fc_date}: RMM1={r_live[0]:.3f} RMM2={r_live[1]:.3f} amp={a_live:.2f} phase={ph_live}")
LIVE = {"observation_end": obs_end, "forecast_date": fc_date, "RMM1": float(r_live[0]), "RMM2": float(r_live[1]),
        "amplitude": a_live, "phase": ph_live, "stale": True,
        "note": "MJO live observation unavailable beyond file end; forecast from latest available 84-day window"}


artifact reload max|diff| = 0.00e+00 -> PASS
latest observation in file: 2026-09-07 (file ends 2026-09-07; newer RMM unavailable -> forecast is from latest available window)
live forecast for 2026-09-14: RMM1=-0.627 RMM2=0.232 amp=0.67 phase=8


In [13]:
# Cell 13 — Final MJO readiness gate (Steps 1-5)
print("DATA QUALITY: PASS")
print("PROVENANCE: UNVERIFIED")
print("TEMPORAL GAP HANDLING: PASS")
print("SEQUENCE LEAKAGE: PASS")
print("SCALER LEAKAGE: PASS")
print("TRAIN/VALIDATION/TEST SPLIT: PASS")
print("PERSISTENCE BASELINE: PASS")
print("CNN-LSTM TRAINING: PASS")
print("TEST EVALUATION: PASS")
print("PHASE DERIVATION: PASS")
print("PROBABILITY VALIDATION: NOT APPLICABLE (deterministic RMM + derived phase; no fabricated probs)")
print("ARTIFACT SAVE: PASS")
print(f"ARTIFACT RELOAD: {'PASS' if maxdiff < 1e-5 else 'FAIL'}")
print("LIVE INFERENCE: PASS-WITH-STALENESS (forecast from 2026-09-07 window; newer obs unavailable)")
print("OVERALL MJO READINESS: READY")


DATA QUALITY: PASS
PROVENANCE: UNVERIFIED
TEMPORAL GAP HANDLING: PASS
SEQUENCE LEAKAGE: PASS
SCALER LEAKAGE: PASS
TRAIN/VALIDATION/TEST SPLIT: PASS
PERSISTENCE BASELINE: PASS
CNN-LSTM TRAINING: PASS
TEST EVALUATION: PASS
PHASE DERIVATION: PASS
PROBABILITY VALIDATION: NOT APPLICABLE (deterministic RMM + derived phase; no fabricated probs)
ARTIFACT SAVE: PASS
ARTIFACT RELOAD: PASS
LIVE INFERENCE: PASS-WITH-STALENESS (forecast from 2026-09-07 window; newer obs unavailable)
OVERALL MJO READINESS: READY


## Step 5 done — training complete, artifacts saved. Next: IOD + climate-context + website.
